# 02 · Modelo de datos

Ensambla las fuentes limpias en una base **SQLite ligera** (`data/processed/vivienda.db`)
para el análisis de la Fase 4. Tres tablas normalizadas —`sniiv`, `segmento_valor`,
`inmuebles`— y el enriquecimiento (créditos + rango de valor) se resuelve por **JOIN**
en `anio` + `segmento`.

Requiere haber corrido antes `01_ingesta.ipynb` (produce los parquet en `data/interim/`).

## Configuración

In [1]:
from pathlib import Path
import sys
import sqlite3

import pandas as pd

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.append(str(REPO / "src"))

from mapeo_segmentos import tabla_segmento_valor
from modelo_datos import construir_sqlite

INTERIM = REPO / "data" / "interim"
PROCESSED = REPO / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

## Insumos

In [2]:
sniiv = pd.read_parquet(INTERIM / "sniiv_merida_tidy.parquet")
inmuebles = pd.read_parquet(INTERIM / "inmuebles_tidy.parquet")
segmento_valor = tabla_segmento_valor()

print("sniiv:         ", sniiv.shape)
print("segmento_valor:", segmento_valor.shape)
print("inmuebles:     ", inmuebles.shape)

sniiv:          (77, 4)
segmento_valor: (66, 6)
inmuebles:      (60, 15)


## Construcción de la base

In [3]:
ruta_db = construir_sqlite(PROCESSED / "vivienda.db", sniiv, segmento_valor, inmuebles)
print("Base creada:", ruta_db.relative_to(REPO))

Base creada: data/processed/vivienda.db


## Verificación — tablas y JOIN

El JOIN enriquece cada crédito con el rango de valor de su segmento. "No disponible"
queda con rango nulo (no participa del análisis de valor).

In [4]:
con = sqlite3.connect(ruta_db)

tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)["name"].tolist()
print("Tablas:", tablas)

consulta = '''
SELECT s.anio, s.segmento, s.acciones, s.monto, v.valor_inf, v.valor_sup
FROM sniiv s
LEFT JOIN segmento_valor v ON s.anio = v.anio AND s.segmento = v.segmento
WHERE s.anio = 2025
ORDER BY v.valor_inf
'''
pd.read_sql(consulta, con)

Tablas: ['sniiv', 'segmento_valor', 'inmuebles']


,anio,segmento,acciones,monto,valor_inf,valor_sup
0,2025,No disponible,1933,0.000000e+00,NaN,NaN
1,2025,Económica,18,5.587524e+06,0.00,405856.28
2,2025,Popular,152,8.521275e+07,405856.28,687892.00
3,2025,Tradicional,1797,1.137877e+09,687892.00,1203811.00
4,2025,Media,1775,1.896407e+09,1203811.00,2579595.00
5,2025,Residencial,268,3.652004e+08,2579595.00,5159190.00
6,2025,Residencial plus,24,9.499127e+06,5159190.00,NaN


## Chequeos

In [5]:
total = pd.read_sql("SELECT SUM(acciones) AS t FROM sniiv", con)["t"][0]
nd = pd.read_sql('''
    SELECT v.valor_inf FROM sniiv s
    LEFT JOIN segmento_valor v ON s.anio=v.anio AND s.segmento=v.segmento
    WHERE s.segmento = "No disponible"
''', con)
n_inm = pd.read_sql("SELECT COUNT(*) AS c FROM inmuebles", con)["c"][0]

print(f"Total acciones en la BD: {int(total):,}")
assert int(total) == 69_448
assert nd["valor_inf"].isna().all(), "No disponible no debe tener rango"
print(f"'No disponible' sin rango: OK")
print(f"Inmuebles: {n_inm} anuncios")
con.close()
print("Modelo de datos: OK")

Total acciones en la BD: 69,448
'No disponible' sin rango: OK
Inmuebles: 60 anuncios
Modelo de datos: OK
